# 01 · The scheduling problem and its synthetic data

SSJ's **Auto-Suggest** asks an engine to choose a **team** and a **start date** for each selected document. This notebook defines that problem, generates a synthetic instance of it, and measures the public stub engine as the starting point.

All data here is generated by `research/synthetic.py`. No real company data is used.

In [ ]:
import sys
from pathlib import Path

# Make the repository root importable, wherever Jupyter was started from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research').is_dir())
sys.path.insert(0, str(ROOT))

from datetime import timedelta

import matplotlib.pyplot as plt
import pandas as pd

from apps.scheduling.engine.pipeline import run_engine
from research.evaluate import score, summary, validate
from research.synthetic import Scenario, generate

## The problem

Each **task** (an SVC, PRJ or XPRJ document) needs `remaining_days` of work.

**Hard rules** (a schedule that breaks one is invalid):

- A team is **2 different technicians**, the same team for every day of the task.
- The work days are **consecutive calendar days**, Sundays and holidays included.
- Work starts on or after the task's `earliest_start`.
- No technician works **two jobs on one day**, counting the jobs already on the calendar.

**Soft goals** (points; higher is better), from the original engine's weights:

| Rewarded | Penalised |
|:---|:---|
| Urgent and XPRJ documents (+50, normal +10) | Each technician-day on a Sunday or holiday (−100) |
| Documents that have waited long (+3 per day, up to +60) | Starting on a Saturday (−30 per technician) |
| Technicians who know the client (+20 each) | Each day of delay after `earliest_start` (−40, urgent −100) |
| Experience with the document type (+5 per job, up to +30) | Each task left unscheduled (−500) |
| Pairs who often work together (+15 per job, up to +45) | |
| Yesterday's job nearby (up to +20 within 15 km) | |

In [ ]:
data = generate(seed=1, scenario=Scenario())

print(f"{len(data['tasks'])} tasks, {len(data['technicians'])} technicians, start {data['start_date']}")
print(f"{len(data['existing_jobs'])} jobs already on the calendar")
print('Holidays:', sorted(data['holidays']))

## The tasks

In [ ]:
tasks = pd.DataFrame(data['tasks'])
tasks[['doc_no', 'doc_type', 'client_name', 'priority_rank', 'initial_lapse_days',
       'remaining_days', 'is_continuation', 'earliest_start']]

In [ ]:
demand = tasks['remaining_days'].sum() * 2
capacity = len(data['technicians']) * 14
print(f'Work asked for: {demand} technician-days; capacity in 14 days: {capacity} technician-days')
tasks.groupby('doc_type')['remaining_days'].describe()

## Where the technicians already are

Grey bars are jobs already on the calendar, red lines are holidays, and the shaded columns are Sundays.

In [ ]:
start = data['start_date']
row = {tech['id']: n for n, tech in enumerate(data['technicians'])}
fig, ax = plt.subplots(figsize=(10, 4))
for job in data['existing_jobs']:
    offset = (job['start_date'] - start).days
    ax.barh(row[job['tech_id']], job['days_duration'], left=offset, color='#94a3b8')
for day in range(-5, 15):
    if (start + timedelta(days=day)).weekday() == 6:
        ax.axvspan(day, day + 1, color='#fde68a', alpha=0.4)
for holiday in data['holidays']:
    ax.axvline((holiday - start).days + 0.5, color='#dc2626')
ax.set_yticks(list(row.values()))
ax.set_yticklabels([tech['name'] for tech in data['technicians']])
ax.set_xlim(-5, 14)
ax.set_xlabel(f'days from {start}')
ax.invert_yaxis()
plt.show()

## Where the clients are

In [ ]:
located = tasks[tasks['location'].notna()].copy()
located['lat'] = located['location'].apply(lambda point: point[0])
located['lng'] = located['location'].apply(lambda point: point[1])
colors = {'svc': '#3b82f6', 'xprj': '#f59e0b', 'prj': '#10b981'}
fig, ax = plt.subplots(figsize=(6, 6))
for doc_type, group in located.groupby('doc_type'):
    ax.scatter(group['lng'], group['lat'], s=40 + group['remaining_days'] * 15,
               color=colors[doc_type], label=doc_type.upper(), alpha=0.7)
ax.set_xlabel('longitude')
ax.set_ylabel('latitude')
ax.set_aspect('equal')
ax.legend()
plt.show()

## The starting point: the public stub

The stub schedules nothing. It is valid, but every task costs the unscheduled penalty. Every algorithm in the next notebooks is measured with the same `validate`, `score` and `summary`.

In [ ]:
stub = run_engine(**data)
print('Problems:', validate(stub, data) or 'none')
print('Score:', score(stub, data).total)
summary(stub, data)

## Next

`02_greedy_baseline.ipynb`: place the tasks one by one, in priority order, with the best free team on the earliest good day.